# 11 — Election data candidate processing

This notebook normalises the House of Commons Library Local Election Handbook candidate workbooks into a single candidate-level table:

`local_election_results_raw_v1.csv`

It also creates or updates the review dictionaries:

- `party_label_dictionary_v1.csv`
- `ward_name_dictionary_v1.csv`
- `ward_name_matching_review_v1.csv`

The HoC files are expected to be national workbooks stored directly in:

`data/raw/election_results`

## 11.1 Project paths and expected files

Run this notebook from the `notebooks` folder inside:

`C:\Users\keena\Documents\Electoral_Tribes\notebooks`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

RAW_ELECTION_DIR = PROJECT_DIR / "data" / "raw" / "election_results"
INTERIM_DIR = PROJECT_DIR / "data" / "interim" / "election_results"
CANDIDATE_INTERIM_DIR = INTERIM_DIR / "candidates_by_year"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "election_results"
DICTIONARY_DIR = PROJECT_DIR / "data" / "dictionaries"
GEOGRAPHY_DIR = PROJECT_DIR / "data" / "geography"

for d in [INTERIM_DIR, CANDIDATE_INTERIM_DIR, PROCESSED_DIR, DICTIONARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = {
    2021: {
        "filename": "local_elections_2021_results-2.xlsx",
        "candidate_sheet": "Candidates-results",
        "candidate_header": 1,
    },
    2022: {
        "filename": "local-elections-2022.xlsx",
        "candidate_sheet": "Candidates-results",
        "candidate_header": 1,
    },
    2023: {
        "filename": "LEH-Candidates-2023.xlsx",
        "candidate_sheet": "Cand_Table",
        "candidate_header": 0,
    },
    2024: {
        "filename": "LEH-2024-results-HoC-version.xlsx",
        "candidate_sheet": "Candidates results",
        "candidate_header": 1,
    },
    2025: {
        "filename": "LEH-2025-results-HoC.xlsx",
        "candidate_sheet": "Candidates result",
        "candidate_header": 1,
    },
}

ELECTION_DATES = {
    2021: "2021-05-06",
    2022: "2022-05-05",
    2023: "2023-05-04",
    2024: "2024-05-02",
    2025: "2025-05-01",
}

print("Project directory:", PROJECT_DIR)
print("Raw election directory:", RAW_ELECTION_DIR)
print("Processed election directory:", PROCESSED_DIR)
print("Dictionary directory:", DICTIONARY_DIR)

missing = []
for year, spec in SOURCE_FILES.items():
    path = RAW_ELECTION_DIR / spec["filename"]
    if not path.exists():
        missing.append(str(path))

if missing:
    print("Missing source files:")
    for m in missing:
        print(" -", m)
    raise FileNotFoundError("One or more HoC source files are missing. Place them in data/raw/election_results.")
else:
    print("All expected source files found.")

Project directory: c:\Users\keena\Documents\Electoral_Tribes
Raw election directory: c:\Users\keena\Documents\Electoral_Tribes\data\raw\election_results
Processed election directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results
Dictionary directory: c:\Users\keena\Documents\Electoral_Tribes\data\dictionaries
All expected source files found.


## 11.2 Helper functions

These functions standardise strings, booleans, party labels, geography types, and result-area keys.

The result-area key deliberately includes year, council, ward code and ward name. This avoids known issues where some source files contain repeated or problematic ward codes.

In [6]:
def clean_colname(col):
    return str(col).strip()


def clean_str(value):
    if pd.isna(value):
        return pd.NA
    return str(value).strip()


def norm_key(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().upper()
    value = value.replace("&", "AND")
    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def to_bool(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        if pd.isna(value):
            return pd.NA
        return bool(int(value))

    text = str(value).strip().lower()

    if text in ["true", "t", "yes", "y", "1", "elected", "winner"]:
        return True
    if text in ["false", "f", "no", "n", "0", ""]:
        return False

    return pd.NA


def classify_geography_type(code):
    if pd.isna(code) or str(code).strip() == "":
        return "name_only_no_ons_code"

    code = str(code).strip().upper()

    if code.startswith("E58") or code.startswith("W58"):
        return "county_electoral_division"

    if code.startswith("E05") or code.startswith("W05"):
        return "electoral_ward_or_division"

    return "unknown_code_type"


def atlas_join_strategy(code, boundary_year):
    geography_type = classify_geography_type(code)

    if geography_type == "county_electoral_division":
        return "needs_county_electoral_division_geography"

    if geography_type == "electoral_ward_or_division" and int(boundary_year) == 2025:
        return "wd25_direct_candidate"

    if geography_type == "electoral_ward_or_division":
        return "needs_historical_ward_crosswalk"

    if geography_type == "name_only_no_ons_code":
        return "needs_ward_name_matching"

    return "manual_review"


def make_result_area_key(source_year, council_name, ward_name, ward_code):
    council_key = norm_key(council_name)
    ward_key = norm_key(ward_name)

    if pd.notna(ward_code) and str(ward_code).strip():
        return f"{source_year}|CODE|{council_key}|{str(ward_code).strip().upper()}|{ward_key}"

    return f"{source_year}|NAME|{council_key}|{ward_key}"


COMMON_PARTY_MAP = {
    "CONSERVATIVE AND UNIONIST PARTY": ("Conservative", "Conservative"),
    "CONSERVATIVE PARTY": ("Conservative", "Conservative"),
    "CON": ("Conservative", "Conservative"),

    "LABOUR PARTY": ("Labour", "Labour"),
    "LABOUR AND CO OPERATIVE PARTY": ("Labour", "Labour"),
    "LABOUR AND CO-OPERATIVE PARTY": ("Labour", "Labour"),
    "LAB": ("Labour", "Labour"),

    "LIBERAL DEMOCRATS": ("Liberal Democrat", "Liberal Democrat"),
    "LIBERAL DEMOCRAT": ("Liberal Democrat", "Liberal Democrat"),
    "LD": ("Liberal Democrat", "Liberal Democrat"),

    "GREEN PARTY": ("Green", "Green"),
    "GREEN": ("Green", "Green"),

    "REFORM UK": ("Reform UK", "Reform / UKIP / Brexit"),
    "REF": ("Reform UK", "Reform / UKIP / Brexit"),

    "UK INDEPENDENCE PARTY": ("UKIP", "Reform / UKIP / Brexit"),
    "UKIP": ("UKIP", "Reform / UKIP / Brexit"),
    "BREXIT PARTY": ("Brexit Party", "Reform / UKIP / Brexit"),

    "SDP": ("SDP", "SDP"),
    "SOCIAL DEMOCRATIC PARTY": ("SDP", "SDP"),

    "INDEPENDENT": ("Independent", "Independent / Localist"),
    "IND": ("Independent", "Independent / Localist"),
    "RESIDENTS": ("Residents", "Independent / Localist"),
}

GENERIC_PARTY_GROUPS = {"", "NAN", "NONE", "OTH", "OTHER", "OTHERS"}


def infer_standard_party(raw_label, party_group=None):
    raw_norm = norm_key(raw_label).replace("_", " ")
    group_norm = norm_key(party_group).replace("_", " ")

    if raw_norm in COMMON_PARTY_MAP:
        return COMMON_PARTY_MAP[raw_norm]

    if group_norm in COMMON_PARTY_MAP:
        return COMMON_PARTY_MAP[group_norm]

    if group_norm and group_norm not in GENERIC_PARTY_GROUPS:
        return (str(party_group).strip(), str(party_group).strip())

    if pd.notna(raw_label) and str(raw_label).strip():
        return (str(raw_label).strip(), "Other")

    return ("Unknown", "Other")


def split_2023_candidate_name(name):
    if pd.isna(name):
        return (pd.NA, pd.NA)

    name = str(name).strip()

    if "," in name:
        last, first = [part.strip() for part in name.split(",", 1)]
        return (first, last)

    parts = name.split()
    if len(parts) >= 2:
        return (" ".join(parts[:-1]), parts[-1])

    return (pd.NA, name)

## 11.3 Candidate-sheet normalisers by year

Each HoC workbook has a slightly different layout. These functions convert each year into one common candidate-level schema.

In [9]:
def normalise_candidate_sheet(year, spec):
    path = RAW_ELECTION_DIR / spec["filename"]

    df = pd.read_excel(
        path,
        sheet_name=spec["candidate_sheet"],
        header=spec["candidate_header"],
    )

    df.columns = [clean_colname(c) for c in df.columns]
    df = df.dropna(how="all").copy()

    if year == 2021:
        out = pd.DataFrame({
            "source_year": year,
            "election_date": ELECTION_DATES[year],
            "election_year": df["Year"],
            "election_type": pd.NA,
            "ordinary_or_by_election": "ordinary",
            "source_file": path.name,
            "source_sheet": spec["candidate_sheet"],

            "county_name": df.get("County name"),
            "council_name": df["Local authority name"],
            "lad_code": df["Local authority code"],
            "upper_tier_authority": pd.NA,
            "lower_tier_authority": df["Local authority name"],

            "ward_name": df["Ward/ED name"],
            "ward_code": df["Ward/ED code"],
            "ec_ward_code": pd.NA,
            "boundary_year": 2021,

            "seats_available": pd.NA,
            "electorate": pd.NA,
            "turnout": pd.NA,
            "valid_votes": df["Total valid votes"],
            "ballots": pd.NA,
            "invalid_votes": pd.NA,

            "candidate_number": df["Candidate number"],
            "candidate_name": df["Candidate name"],
            "candidate_first_names": pd.NA,
            "candidate_last_names": pd.NA,
            "candidate_gender": df["Candidate gender"],
            "incumbent": df["Inumbent"],

            "raw_party_label": df["Party name"],
            "party_group": df["Party group"],
            "party_id": df["Party ID"],

            "votes": df["Votes"],
            "vote_share": df["Votes (%)"],
            "elected": df["Elected"],
            "rank": pd.NA,
            "votes_effective": df["Vote effective"],
        })

    elif year == 2022:
        county_name = (
            df["COUNTYNAME"].combine_first(df["County name"])
            if "COUNTYNAME" in df.columns and "County name" in df.columns
            else df.get("COUNTYNAME", df.get("County name"))
        )

        out = pd.DataFrame({
            "source_year": year,
            "election_date": ELECTION_DATES[year],
            "election_year": df["Year"],
            "election_type": df["Type"],
            "ordinary_or_by_election": "ordinary",
            "source_file": path.name,
            "source_sheet": spec["candidate_sheet"],

            "county_name": county_name,
            "council_name": df["Local authority name"],
            "lad_code": df["Local authority code"],
            "upper_tier_authority": pd.NA,
            "lower_tier_authority": df["Local authority name"],

            "ward_name": df["Ward name"],
            "ward_code": df["Ward code"],
            "ec_ward_code": pd.NA,
            "boundary_year": 2022,

            "seats_available": df["Vacancies"],
            "electorate": pd.NA,
            "turnout": pd.NA,
            "valid_votes": df["Total valid votes"],
            "ballots": pd.NA,
            "invalid_votes": pd.NA,

            "candidate_number": df["Candidate number"],
            "candidate_name": df["Candidate name"],
            "candidate_first_names": pd.NA,
            "candidate_last_names": pd.NA,
            "candidate_gender": df["Candidate gender"],
            "incumbent": df["Incumbent"],

            "raw_party_label": df["Party name"],
            "party_group": df["Party group"],
            "party_id": df["Party ID"],

            "votes": df["Votes"],
            "vote_share": pd.NA,
            "elected": df["Elected"],
            "rank": pd.NA,
            "votes_effective": df["Vote effective"],
        })

    elif year == 2023:
        split_names = df["NAME"].apply(split_2023_candidate_name)

        out = pd.DataFrame({
            "source_year": year,
            "election_date": ELECTION_DATES[year],
            "election_year": df["YEAR"],
            "election_type": df["TYPE"],
            "ordinary_or_by_election": "ordinary",
            "source_file": path.name,
            "source_sheet": spec["candidate_sheet"],

            "county_name": df["COUNTYNAME"],
            "council_name": df["DISTRICTNAME"],
            "lad_code": pd.NA,
            "upper_tier_authority": pd.NA,
            "lower_tier_authority": df["DISTRICTNAME"],

            "ward_name": df["WARDNAME"],
            "ward_code": pd.NA,
            "ec_ward_code": pd.NA,
            "boundary_year": 2023,

            "seats_available": df["VACS"],
            "electorate": df["ELECT"],
            "turnout": df["TURNOUT"],
            "valid_votes": df["EFFECTIVEVOTES"],
            "ballots": pd.NA,
            "invalid_votes": pd.NA,

            "candidate_number": df["NUMCAND"],
            "candidate_name": df["NAME"],
            "candidate_first_names": [x[0] for x in split_names],
            "candidate_last_names": [x[1] for x in split_names],
            "candidate_gender": df["GENDER"],
            "incumbent": df["INCUMB"],

            "raw_party_label": df["PARTYNAME"],
            "party_group": df["PARTYGROUP"],
            "party_id": df["PARTYID"],

            "votes": df["VOTE"],
            "vote_share": pd.NA,
            "elected": df["WINNER"],
            "rank": pd.NA,
            "votes_effective": df["VOTEEFFECTIVE"],
        })

    elif year == 2024:
        out = pd.DataFrame({
            "source_year": year,
            "election_date": ELECTION_DATES[year],
            "election_year": year,
            "election_type": df["Election type"],
            "ordinary_or_by_election": "ordinary",
            "source_file": path.name,
            "source_sheet": spec["candidate_sheet"],

            "county_name": pd.NA,
            "council_name": df["Local authority name"],
            "lad_code": df["ONS local authority code"],
            "upper_tier_authority": pd.NA,
            "lower_tier_authority": df["Local authority name"],

            "ward_name": df["Ward name"],
            "ward_code": df["Ward code"],
            "ec_ward_code": pd.NA,
            "boundary_year": 2024,

            "seats_available": df["Vacancies"],
            "electorate": df["Electorate"],
            "turnout": df["Turnout (%)"],
            "valid_votes": df["Total votes"],
            "ballots": pd.NA,
            "invalid_votes": pd.NA,

            "candidate_number": df["Candidate number"],
            "candidate_name": df["Name"],
            "candidate_first_names": pd.NA,
            "candidate_last_names": pd.NA,
            "candidate_gender": df["Gender"],
            "incumbent": df["Incumbent"],

            "raw_party_label": df["Party name"],
            "party_group": df["Party Group"],
            "party_id": df["Party ID"],

            "votes": df["Votes"],
            "vote_share": pd.NA,
            "elected": df["Elected"],
            "rank": pd.NA,
            "votes_effective": df["Vote effective"],
        })

    elif year == 2025:
        out = pd.DataFrame({
            "source_year": year,
            "election_date": ELECTION_DATES[year],
            "election_year": year,
            "election_type": df["Election type"],
            "ordinary_or_by_election": "ordinary",
            "source_file": path.name,
            "source_sheet": spec["candidate_sheet"],

            "county_name": pd.NA,
            "council_name": df["Lower tier authority"],
            "lad_code": pd.NA,
            "upper_tier_authority": df["Upper tier authority"],
            "lower_tier_authority": df["Lower tier authority"],

            "ward_name": df["Ward/ County Electoral District name"],
            "ward_code": df["ONS ward code"],
            "ec_ward_code": df["EC ward code"],
            "boundary_year": 2025,

            "seats_available": df["Seats contested"],
            "electorate": pd.NA,
            "turnout": pd.NA,
            "valid_votes": pd.NA,
            "ballots": pd.NA,
            "invalid_votes": pd.NA,

            "candidate_number": pd.NA,
            "candidate_name": df["Candidate name"],
            "candidate_first_names": df["First names"],
            "candidate_last_names": df["Last names"],
            "candidate_gender": df["Gender"],
            "incumbent": df["Incumbent"],

            "raw_party_label": df["Party name"],
            "party_group": pd.NA,
            "party_id": pd.NA,

            "votes": df["Votes cast"],
            "vote_share": pd.NA,
            "elected": df["Elected"],
            "rank": df["Rank"],
            "votes_effective": df["Votes effective"],
        })

    else:
        raise ValueError(f"No normaliser defined for {year}")

    # Standardise common field types
    string_cols = [
        "county_name", "council_name", "lad_code", "upper_tier_authority", "lower_tier_authority",
        "ward_name", "ward_code", "ec_ward_code", "candidate_name", "candidate_first_names",
        "candidate_last_names", "candidate_gender", "raw_party_label", "party_group"
    ]

    for col in string_cols:
        if col in out.columns:
            out[col] = out[col].map(clean_str)

    numeric_cols = [
        "election_year", "boundary_year", "seats_available", "electorate", "turnout",
        "valid_votes", "ballots", "invalid_votes", "candidate_number", "votes",
        "vote_share", "rank"
    ]

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    bool_cols = ["elected", "incumbent", "votes_effective"]

    for col in bool_cols:
        out[col] = out[col].map(to_bool)

    out["geography_type"] = out["ward_code"].map(classify_geography_type)

    out["atlas_join_strategy"] = [
        atlas_join_strategy(code, boundary_year)
        for code, boundary_year in zip(out["ward_code"], out["boundary_year"])
    ]

    out["result_area_key"] = [
        make_result_area_key(year, council, ward, code)
        for year, council, ward, code in zip(
            out["source_year"], out["council_name"], out["ward_name"], out["ward_code"]
        )
    ]

    party_suggestions = [
        infer_standard_party(raw, group)
        for raw, group in zip(out["raw_party_label"], out["party_group"])
    ]

    out["standard_party_label"] = [x[0] for x in party_suggestions]
    out["party_family"] = [x[1] for x in party_suggestions]

    out["source_url"] = pd.NA
    out["source_notes"] = "House of Commons Library Local Election Handbook workbook"

    out["data_quality_flag"] = np.where(
        out["ward_code"].isna(),
        "needs_ward_code_match",
        "ok"
    )

    out["manual_review_required"] = (
        out["ward_code"].isna()
        | out["raw_party_label"].isna()
        | out["geography_type"].eq("unknown_code_type")
    )

    # Deterministic row identifier
    key_cols = [
    "source_year",
    "result_area_key",
    "candidate_number",
    "candidate_name",
    "raw_party_label",
    "votes",
    ]   

    # Ensure all key columns exist
    for col in key_cols:
        if col not in out.columns:
         out[col] = ""

    # Build a stable row-level key defensively.
    # This avoids pandas join errors when values are float, NaN, None, etc.
    key_text = (
        out[key_cols]
        .fillna("")
        .apply(lambda row: "|".join(str(value) for value in row.values), axis=1)
    )

    # If rerunning the function during notebook development, avoid duplicate insertion.
    if "result_id" in out.columns:
        out = out.drop(columns=["result_id"])

    out.insert(
        0,
        "result_id",
        [hashlib.md5(text.encode("utf-8")).hexdigest()[:16] for text in key_text]
    )

    return out

## 11.4 Normalise all candidate sheets

In [10]:
candidate_frames = []

for year, spec in SOURCE_FILES.items():
    print(f"Processing {year}...")
    year_df = normalise_candidate_sheet(year, spec)

    out_path = CANDIDATE_INTERIM_DIR / f"candidates_{year}_normalised_v1.csv"
    year_df.to_csv(out_path, index=False)

    print("  rows:", len(year_df))
    print("  result areas:", year_df["result_area_key"].nunique())
    print("  geography types:", year_df["geography_type"].value_counts(dropna=False).to_dict())
    print("  saved:", out_path)

    candidate_frames.append(year_df)

candidates = pd.concat(candidate_frames, ignore_index=True)

print("Combined rows:", len(candidates))
print("Combined result areas:", candidates["result_area_key"].nunique())

display(candidates.head())

Processing 2021...
  rows: 18044
  result areas: 3864
  geography types: {'electoral_ward_or_division': 12221, 'county_electoral_division': 5823}
  saved: c:\Users\keena\Documents\Electoral_Tribes\data\interim\election_results\candidates_by_year\candidates_2021_normalised_v1.csv
Processing 2022...
  rows: 18481
  result areas: 3610
  geography types: {'electoral_ward_or_division': 18481}
  saved: c:\Users\keena\Documents\Electoral_Tribes\data\interim\election_results\candidates_by_year\candidates_2022_normalised_v1.csv
Processing 2023...
  rows: 25697
  result areas: 4831
  geography types: {'name_only_no_ons_code': 25697}
  saved: c:\Users\keena\Documents\Electoral_Tribes\data\interim\election_results\candidates_by_year\candidates_2023_normalised_v1.csv
Processing 2024...
  rows: 10029
  result areas: 1903
  geography types: {'electoral_ward_or_division': 10029}
  saved: c:\Users\keena\Documents\Electoral_Tribes\data\interim\election_results\candidates_by_year\candidates_2024_normalis

,result_id,source_year,election_date,election_year,election_type,ordinary_or_by_election,source_file,source_sheet,county_name,council_name,...,votes_effective,geography_type,atlas_join_strategy,result_area_key,standard_party_label,party_family,source_url,source_notes,data_quality_flag,manual_review_required
0,364217414feb2f58,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,True,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,Green,Green,<NA>,House of Commons Library Local Election Handbo...,ok,False
1,40f69ad91cd92fb8,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,True,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,Labour,Labour,<NA>,House of Commons Library Local Election Handbo...,ok,False
2,cd14a3fbf2576559,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,False,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,Green,Green,<NA>,House of Commons Library Local Election Handbo...,ok,False
3,0acd28cac5c5be06,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,False,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,Green,Green,<NA>,House of Commons Library Local Election Handbo...,ok,False
4,6b848cda8054856f,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,False,electoral_ward_or_division,needs_historical_ward_crosswalk,2021|CODE|BRISTOL_CITY_OF|E05010885|ASHLEY,Labour,Labour,<NA>,House of Commons Library Local Election Handbo...,ok,False


## 11.5 Create or update the party label dictionary

Review this file manually if you want to standardise smaller parties, residents' groups, and local independents before final modelling.

In [11]:
PARTY_DICT_PATH = DICTIONARY_DIR / "party_label_dictionary_v1.csv"

new_party_rows = (
    candidates[["raw_party_label", "party_group", "party_id", "standard_party_label", "party_family"]]
    .drop_duplicates()
    .sort_values(["raw_party_label", "party_group"], na_position="last")
)

new_party_dict = (
    new_party_rows
    .groupby(["raw_party_label"], dropna=False, as_index=False)
    .agg(
        suggested_standard_party_label=("standard_party_label", "first"),
        suggested_party_family=("party_family", "first"),
        source_party_groups=("party_group", lambda s: "; ".join(sorted({str(x) for x in s.dropna().unique()}))[:500]),
        source_party_ids=("party_id", lambda s: "; ".join(sorted({str(x) for x in s.dropna().unique()}))[:500]),
        occurrences=("raw_party_label", "size")
    )
)

new_party_dict["standard_party_label"] = new_party_dict["suggested_standard_party_label"]
new_party_dict["party_family"] = new_party_dict["suggested_party_family"]
new_party_dict["review_status"] = np.where(new_party_dict["party_family"].eq("Other"), "review", "suggested")
new_party_dict["notes"] = ""

new_party_dict = new_party_dict[
    [
        "raw_party_label",
        "standard_party_label",
        "party_family",
        "review_status",
        "source_party_groups",
        "source_party_ids",
        "occurrences",
        "notes",
    ]
]

if PARTY_DICT_PATH.exists():
    existing = pd.read_csv(PARTY_DICT_PATH, low_memory=False)

    combined = existing.merge(
        new_party_dict,
        on="raw_party_label",
        how="outer",
        suffixes=("", "_new")
    )

    for col in ["standard_party_label", "party_family", "review_status", "source_party_groups", "source_party_ids", "occurrences", "notes"]:
        new_col = f"{col}_new"
        if new_col in combined.columns:
            combined[col] = combined[col].combine_first(combined[new_col])
            combined = combined.drop(columns=[new_col])

    party_dict = combined
else:
    party_dict = new_party_dict

party_dict.to_csv(PARTY_DICT_PATH, index=False)

print("Saved party dictionary:", PARTY_DICT_PATH)
print("Rows:", len(party_dict))
display(party_dict.head(20))

Saved party dictionary: c:\Users\keena\Documents\Electoral_Tribes\data\dictionaries\party_label_dictionary_v1.csv
Rows: 398


,raw_party_label,standard_party_label,party_family,review_status,source_party_groups,source_party_ids,occurrences,notes
0,ABTV,ABTV,Other,review,OTH,401; 402,2,
1,ADF,ADF,Other,review,OTH,401; 402; 403; 404; 406,5,
2,AEF,AEF,Other,review,OTH,401,1,
3,AFP,AFP,Other,review,OTH,401; 402,2,
4,AFW,AFW,Other,review,OTH,402; 403,2,
5,AGS,AGS,Other,review,OTH,402; 403,2,
6,AIAR,AIAR,Other,review,OTH,401,1,
7,AN IND,AN IND,Other,review,OTH,402,1,
8,AND IND,AND IND,Other,review,OTH,401; 402,2,
9,ASH IND,ASH IND,Other,review,OTH,401; 402; 403,3,


## 11.6 Create or update ward-name dictionary and review file

In [12]:
WARD_DICT_PATH = DICTIONARY_DIR / "ward_name_dictionary_v1.csv"
WARD_REVIEW_PATH = DICTIONARY_DIR / "ward_name_matching_review_v1.csv"

new_ward_dict = (
    candidates[
        [
            "source_year", "boundary_year", "county_name", "council_name", "lad_code",
            "ward_name", "ward_code", "ec_ward_code", "geography_type", "atlas_join_strategy"
        ]
    ]
    .drop_duplicates()
    .sort_values(["source_year", "council_name", "ward_name"], na_position="last")
)

new_ward_dict["raw_ward_name"] = new_ward_dict["ward_name"]
new_ward_dict["standard_ward_name"] = new_ward_dict["ward_name"]
new_ward_dict["matched_wd25cd"] = ""
new_ward_dict["matched_wd25nm"] = ""
new_ward_dict["matched_lad25cd"] = ""
new_ward_dict["matched_lad25nm"] = ""

new_ward_dict["review_status"] = np.where(
    new_ward_dict["atlas_join_strategy"].isin([
        "needs_ward_name_matching",
        "needs_county_electoral_division_geography",
        "manual_review"
    ]),
    "review",
    "suggested"
)

new_ward_dict["notes"] = ""

new_ward_dict = new_ward_dict[
    [
        "source_year",
        "boundary_year",
        "county_name",
        "council_name",
        "lad_code",
        "raw_ward_name",
        "standard_ward_name",
        "ward_code",
        "ec_ward_code",
        "geography_type",
        "atlas_join_strategy",
        "matched_wd25cd",
        "matched_wd25nm",
        "matched_lad25cd",
        "matched_lad25nm",
        "review_status",
        "notes",
    ]
]

if WARD_DICT_PATH.exists():
    existing = pd.read_csv(WARD_DICT_PATH, low_memory=False)

    key_cols = ["source_year", "council_name", "raw_ward_name", "ward_code", "boundary_year"]

    combined = existing.merge(
        new_ward_dict,
        on=key_cols,
        how="outer",
        suffixes=("", "_new")
    )

    for col in new_ward_dict.columns:
        if col in key_cols:
            continue
        new_col = f"{col}_new"
        if new_col in combined.columns:
            combined[col] = combined[col].combine_first(combined[new_col])
            combined = combined.drop(columns=[new_col])

    ward_dict = combined
else:
    ward_dict = new_ward_dict

ward_dict.to_csv(WARD_DICT_PATH, index=False)

review = ward_dict[
    ward_dict["atlas_join_strategy"].isin([
        "needs_ward_name_matching",
        "needs_county_electoral_division_geography",
        "manual_review"
    ])
].copy()

review["suggested_action"] = np.select(
    [
        review["atlas_join_strategy"].eq("needs_ward_name_matching"),
        review["atlas_join_strategy"].eq("needs_county_electoral_division_geography"),
        review["atlas_join_strategy"].eq("manual_review"),
    ],
    [
        "Match raw ward/council name to an official ward code or reviewed standard name.",
        "Do not join directly to WD25. Use county electoral division geography/crosswalk.",
        "Review code/name manually.",
    ],
    default="Review manually."
)

review.to_csv(WARD_REVIEW_PATH, index=False)

print("Saved ward dictionary:", WARD_DICT_PATH)
print("Saved ward matching review:", WARD_REVIEW_PATH)
print("Ward dictionary rows:", len(ward_dict))
print("Review rows:", len(review))

display(review.head(20))

Saved ward dictionary: c:\Users\keena\Documents\Electoral_Tribes\data\dictionaries\ward_name_dictionary_v1.csv
Saved ward matching review: c:\Users\keena\Documents\Electoral_Tribes\data\dictionaries\ward_name_matching_review_v1.csv
Ward dictionary rows: 15609
Review rows: 7085


,source_year,boundary_year,county_name,council_name,lad_code,raw_ward_name,standard_ward_name,ward_code,ec_ward_code,geography_type,atlas_join_strategy,matched_wd25cd,matched_wd25nm,matched_lad25cd,matched_lad25nm,review_status,notes,suggested_action
5,2021,2021,West Sussex,Adur,E07000223,Lancing,Lancing,E58001636,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...
10,2021,2021,West Sussex,Adur,E07000223,Shoreham North,Shoreham North,E58001653,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...
11,2021,2021,West Sussex,Adur,E07000223,Shoreham South,Shoreham South,E58001654,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...
12,2021,2021,West Sussex,Adur,E07000223,Sompting And North Lancing,Sompting And North Lancing,E58001655,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...
14,2021,2021,West Sussex,Adur,E07000223,Southwick,Southwick,E58001658,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...
20,2021,2021,Derbyshire,Amber Valley,E07000032,Alfreton And Somercotes,Alfreton And Somercotes,E58000193,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...
21,2021,2021,Derbyshire,Amber Valley,E07000032,Alport And Derwent,Alport And Derwent,E58000194,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...
22,2021,2021,Derbyshire,Amber Valley,E07000032,Belper,Belper,E58000199,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...
26,2021,2021,Derbyshire,Amber Valley,E07000032,Duffield And Belper South,Duffield And Belper South,E58000216,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...
27,2021,2021,Derbyshire,Amber Valley,E07000032,Greater Heanor,Greater Heanor,E58000221,<NA>,county_electoral_division,needs_county_electoral_division_geography,,,,,review,,Do not join directly to WD25. Use county elect...


## 11.7 Apply reviewed dictionaries

If you manually edit `party_label_dictionary_v1.csv` or `ward_name_dictionary_v1.csv`, rerun from this cell down.

In [13]:
party_dict = pd.read_csv(PARTY_DICT_PATH, low_memory=False)
ward_dict = pd.read_csv(WARD_DICT_PATH, low_memory=False)

# Apply party dictionary.
party_apply = party_dict[["raw_party_label", "standard_party_label", "party_family"]].drop_duplicates("raw_party_label")

candidates_final = (
    candidates
    .drop(columns=["standard_party_label", "party_family"], errors="ignore")
    .merge(party_apply, on="raw_party_label", how="left", validate="many_to_one")
)

candidates_final["standard_party_label"] = candidates_final["standard_party_label"].fillna(candidates_final["raw_party_label"])
candidates_final["party_family"] = candidates_final["party_family"].fillna("Other")

# Apply ward dictionary standard names, preserving existing ward codes.
ward_apply = ward_dict.rename(columns={"raw_ward_name": "ward_name"})

ward_apply = ward_apply[
    [
        "source_year", "council_name", "ward_name", "ward_code", "boundary_year",
        "standard_ward_name", "matched_wd25cd", "matched_wd25nm",
        "matched_lad25cd", "matched_lad25nm", "review_status"
    ]
].drop_duplicates(
    ["source_year", "council_name", "ward_name", "ward_code", "boundary_year"]
)

candidates_final = candidates_final.merge(
    ward_apply,
    on=["source_year", "council_name", "ward_name", "ward_code", "boundary_year"],
    how="left",
    validate="many_to_one"
)

candidates_final["standard_ward_name"] = candidates_final["standard_ward_name"].fillna(candidates_final["ward_name"])

candidates_final["manual_review_required"] = (
    candidates_final["manual_review_required"].fillna(False).astype(bool)
    | candidates_final["review_status"].eq("review").fillna(False)
)

display(candidates_final.head())

,result_id,source_year,election_date,election_year,election_type,ordinary_or_by_election,source_file,source_sheet,county_name,council_name,...,data_quality_flag,manual_review_required,standard_party_label,party_family,standard_ward_name,matched_wd25cd,matched_wd25nm,matched_lad25cd,matched_lad25nm,review_status
0,364217414feb2f58,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,ok,False,Green,Green,Ashley,NaN,NaN,NaN,NaN,suggested
1,40f69ad91cd92fb8,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,ok,False,Labour,Labour,Ashley,NaN,NaN,NaN,NaN,suggested
2,cd14a3fbf2576559,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,ok,False,Green,Green,Ashley,NaN,NaN,NaN,NaN,suggested
3,0acd28cac5c5be06,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,ok,False,Green,Green,Ashley,NaN,NaN,NaN,NaN,suggested
4,6b848cda8054856f,2021,2021-05-06,2021,<NA>,ordinary,local_elections_2021_results-2.xlsx,Candidates-results,NaN,"Bristol, City Of",...,ok,False,Labour,Labour,Ashley,NaN,NaN,NaN,NaN,suggested


## 11.8 Save final candidate-level election results file

In [14]:
CANDIDATE_OUTPUT = PROCESSED_DIR / "local_election_results_raw_v1.csv"

preferred_order = [
    "result_id",
    "result_area_key",
    "source_year",
    "election_date",
    "election_year",
    "election_type",
    "ordinary_or_by_election",
    "source_file",
    "source_sheet",

    "county_name",
    "council_name",
    "lad_code",
    "upper_tier_authority",
    "lower_tier_authority",

    "ward_name",
    "standard_ward_name",
    "ward_code",
    "ec_ward_code",
    "boundary_year",
    "geography_type",
    "atlas_join_strategy",

    "seats_available",
    "electorate",
    "turnout",
    "valid_votes",
    "ballots",
    "invalid_votes",

    "candidate_number",
    "candidate_name",
    "candidate_first_names",
    "candidate_last_names",
    "candidate_gender",
    "incumbent",

    "raw_party_label",
    "standard_party_label",
    "party_family",
    "party_group",
    "party_id",

    "votes",
    "vote_share",
    "elected",
    "rank",
    "votes_effective",

    "matched_wd25cd",
    "matched_wd25nm",
    "matched_lad25cd",
    "matched_lad25nm",

    "source_url",
    "source_notes",
    "data_quality_flag",
    "manual_review_required",
]

ordered = [c for c in preferred_order if c in candidates_final.columns]
remaining = [c for c in candidates_final.columns if c not in ordered]

candidates_final = candidates_final[ordered + remaining]

candidates_final.to_csv(CANDIDATE_OUTPUT, index=False)

print("Saved:", CANDIDATE_OUTPUT)
print("Rows:", len(candidates_final))
print("Result areas:", candidates_final["result_area_key"].nunique())
print("Manual review rows:", candidates_final["manual_review_required"].sum())

summary = (
    candidates_final
    .groupby(["source_year", "geography_type", "atlas_join_strategy"], as_index=False)
    .agg(
        rows=("result_id", "count"),
        result_areas=("result_area_key", "nunique")
    )
)

display(summary)

summary.to_csv(PROCESSED_DIR / "local_election_results_raw_processing_report_v1.csv", index=False)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\local_election_results_raw_v1.csv
Rows: 80392
Result areas: 15609
Manual review rows: 36394


,source_year,geography_type,atlas_join_strategy,rows,result_areas
0,2021,county_electoral_division,needs_county_electoral_division_geography,5823,1366
1,2021,electoral_ward_or_division,needs_historical_ward_crosswalk,12221,2498
2,2022,electoral_ward_or_division,needs_historical_ward_crosswalk,18481,3610
3,2023,name_only_no_ons_code,needs_ward_name_matching,25697,4831
4,2024,electoral_ward_or_division,needs_historical_ward_crosswalk,10029,1903
5,2025,county_electoral_division,needs_county_electoral_division_geography,4874,888
6,2025,electoral_ward_or_division,wd25_direct_candidate,3267,513


## 11.9 Geography files to locate

Notebook 11 does not require geography lookup files to create the candidate-level table. However, future join work will need historical ward/division geography support.

Place these in `data/geography` as you locate them:

| Purpose | Suggested local filename | Notes |
|---|---|---|
| 2021 ward/division bridge | `oa21_to_wd21_lad21_eng_wal.csv` | For election areas using 2021 ward codes. |
| 2022 ward/division bridge | `oa21_to_wd22_lad22_eng_wal.csv` | For election areas using 2022 ward codes. |
| 2023 ward/division bridge | `oa21_to_wd23_lad23_eng_wal.csv` | Helps once 2023 name-only rows are matched to codes. |
| 2024 ward/division bridge | `oa21_to_wd24_lad24_eng_wal.csv` | For 2024 ward results. |
| 2025 ward bridge | `oa21_to_wd25_lad25_eng_wal(may25).csv` | You already use this in the atlas pipeline. |
| County electoral divisions | `county_electoral_divisions_<year>_boundaries_or_lookup` | Needed for E58* county division rows. Do not join these directly to WD25. |